# Coastal flood step 10: min/max sector return-period comparison

Compares minimum vs maximum scenarios by **sector and return period** for:
- damages without mangroves
- damages with mangroves
- avoided damages
- percent avoided vs no-mangrove baseline

Inputs are loaded from each scenario's `damage_estimates/sector_subsector_return_period_damages_with_avoided_share.csv`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import FuncFormatter, MaxNLocator

base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers")
fx_jmd_per_usd = 150.0  # 1 USD = 150 JMD
mangrove_attribution_buffer_m = 5000
return_period_input_name = "sector_subsector_return_period_damages_with_avoided_share.csv"

scenario_paths = {
    "minimum": base_path / "dphil_paper_3/results/02_damage_estimates/coastal_flood_damages/results_coastal_minimum_scenario/damage_estimates",
    "maximum": base_path / "dphil_paper_3/results/02_damage_estimates/coastal_flood_damages/results_coastal_maximum_scenario/damage_estimates",
}

comparison_output_dir = base_path / "dphil_paper_3/results_coastal_scenario_comparison"
comparison_output_dir.mkdir(parents=True, exist_ok=True)
plot_output_dir = comparison_output_dir / "return_period_sector_charts"
plot_output_dir.mkdir(parents=True, exist_ok=True)


def money_mn_fmt(value, _position):
    if abs(value) >= 100:
        return f"{value:,.0f}"
    return f"{value:,.1f}"


for scenario_name, scenario_path in scenario_paths.items():
    return_period_file = scenario_path / return_period_input_name
    attribution_file = (
        scenario_path
        / "mangrove_attribution"
        / f"attribution_breakdown_by_sector_{mangrove_attribution_buffer_m}m.csv"
    )
    print(scenario_name, "->", return_period_file)
    if not return_period_file.exists():
        raise FileNotFoundError(f"Missing input file: {return_period_file}")
    if not attribution_file.exists():
        raise FileNotFoundError(f"Missing attribution file: {attribution_file}")



In [ ]:
def load_scenario_rp_table(damage_estimates_path: Path, scenario_name: str) -> pd.DataFrame:
    return_period_df = pd.read_csv(damage_estimates_path / return_period_input_name)
    required_cols = [
        "Sector",
        "Subsector",
        "ReturnPeriod",
        "Damages_With_Mangroves_JD",
        "Damages_Without_Mangroves_JD",
        "Avoided_Damages_JD",
    ]
    missing_cols = [column_name for column_name in required_cols if column_name not in return_period_df.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns in {scenario_name}: {missing_cols}")
    return_period_df["Scenario"] = scenario_name
    return return_period_df


def load_scenario_sector_attribution(damage_estimates_path: Path, scenario_name: str, attribution_buffer_m: int) -> pd.DataFrame:
    attribution_path = (
        damage_estimates_path
        / "mangrove_attribution"
        / f"attribution_breakdown_by_sector_{attribution_buffer_m}m.csv"
    )
    sector_attribution_df = pd.read_csv(attribution_path)
    required_cols = ["Sector", "Avoided_EAD_USD", "Attributed_EAD_USD"]
    missing_cols = [column_name for column_name in required_cols if column_name not in sector_attribution_df.columns]
    if missing_cols:
        raise ValueError(f"Missing required attribution columns in {scenario_name}: {missing_cols}")

    sector_attribution_df = sector_attribution_df[required_cols].copy()
    sector_attribution_df["Scenario"] = scenario_name
    sector_attribution_df["Attributed_Share_5000m"] = np.where(
        sector_attribution_df["Avoided_EAD_USD"] != 0,
        sector_attribution_df["Attributed_EAD_USD"] / sector_attribution_df["Avoided_EAD_USD"],
        0.0,
    )
    sector_attribution_df["Attributed_Share_5000m"] = sector_attribution_df["Attributed_Share_5000m"].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return sector_attribution_df


rp_raw = pd.concat(
    [
        load_scenario_rp_table(scenario_path, scenario_name)
        for scenario_name, scenario_path in scenario_paths.items()
    ],
    ignore_index=True,
)

sector_attribution = pd.concat(
    [
        load_scenario_sector_attribution(scenario_path, scenario_name, mangrove_attribution_buffer_m)
        for scenario_name, scenario_path in scenario_paths.items()
    ],
    ignore_index=True,
)

sector_attribution_shares = sector_attribution[["Scenario", "Sector", "Attributed_Share_5000m"]].copy()

rp_raw[["Scenario", "Sector", "Subsector", "ReturnPeriod"]].head()



In [ ]:
sum_cols_jd = [
    "Damages_With_Mangroves_JD",
    "Damages_Without_Mangroves_JD",
    "Avoided_Damages_JD",
]

rp_sector = (
    rp_raw.groupby(["Scenario", "Sector", "ReturnPeriod"], as_index=False)[sum_cols_jd]
    .sum()
)

rp_sector = rp_sector.merge(
    sector_attribution_shares,
    on=["Scenario", "Sector"],
    how="left",
)
rp_sector["Attributed_Share_5000m"] = rp_sector["Attributed_Share_5000m"].fillna(0.0)

# Swap total avoided values to 5000m-attributed avoided values.
rp_sector["Avoided_Damages_JD"] = (
    rp_sector["Avoided_Damages_JD"] * rp_sector["Attributed_Share_5000m"]
)

rp_sector["Percent_Avoided_vs_NoMangroves"] = np.where(
    rp_sector["Damages_Without_Mangroves_JD"] > 0,
    100.0 * rp_sector["Avoided_Damages_JD"] / rp_sector["Damages_Without_Mangroves_JD"],
    np.nan,
)

for column_name in sum_cols_jd:
    rp_sector[column_name.replace("_JD", "_USD")] = rp_sector[column_name] / fx_jmd_per_usd
    rp_sector[column_name.replace("_JD", "_USD_mn")] = rp_sector[column_name] / fx_jmd_per_usd / 1e6

rp_sector = rp_sector.sort_values(["Scenario", "Sector", "ReturnPeriod"]).reset_index(drop=True)

min_sector = (
    rp_sector[rp_sector["Scenario"] == "minimum"]
    .drop(columns=["Scenario"])
    .add_prefix("Min_")
    .rename(columns={"Min_Sector": "Sector", "Min_ReturnPeriod": "ReturnPeriod"})
)

max_sector = (
    rp_sector[rp_sector["Scenario"] == "maximum"]
    .drop(columns=["Scenario"])
    .add_prefix("Max_")
    .rename(columns={"Max_Sector": "Sector", "Max_ReturnPeriod": "ReturnPeriod"})
)

rp_sector_compare = (
    min_sector.merge(max_sector, on=["Sector", "ReturnPeriod"], how="outer")
    .sort_values(["Sector", "ReturnPeriod"])
    .reset_index(drop=True)
)

delta_metrics = [
    "Damages_With_Mangroves_JD",
    "Damages_Without_Mangroves_JD",
    "Avoided_Damages_JD",
    "Damages_With_Mangroves_USD_mn",
    "Damages_Without_Mangroves_USD_mn",
    "Avoided_Damages_USD_mn",
    "Percent_Avoided_vs_NoMangroves",
]
for metric_name in delta_metrics:
    rp_sector_compare[f"Delta_{metric_name}_MaxMinusMin"] = (
        rp_sector_compare[f"Max_{metric_name}"] - rp_sector_compare[f"Min_{metric_name}"]
    )

out_totals = comparison_output_dir / "sector_return_period_totals_by_scenario.csv"
out_compare = comparison_output_dir / "sector_return_period_comparison_minimum_vs_maximum.csv"
out_shares = comparison_output_dir / f"sector_attribution_shares_{mangrove_attribution_buffer_m}m.csv"

rp_sector.to_csv(out_totals, index=False)
rp_sector_compare.to_csv(out_compare, index=False)
sector_attribution_shares.to_csv(out_shares, index=False)

print("Saved:", out_totals)
print("Saved:", out_compare)
print("Saved:", out_shares)
print(f"Avoided damages are using {mangrove_attribution_buffer_m}m-attributed values.")
rp_sector_compare.head(12)



In [ ]:
return_periods = sorted(rp_sector_compare["ReturnPeriod"].dropna().unique())

for return_period in return_periods:
    rp_df = (
        rp_sector_compare[rp_sector_compare["ReturnPeriod"] == return_period]
        .sort_values("Sector")
        .reset_index(drop=True)
    )

    x_positions = np.arange(len(rp_df))
    sectors = rp_df["Sector"].astype(str)
    bar_width = 0.24

    fig, axes = plt.subplots(1, 3, figsize=(20, 5), constrained_layout=True)

    # Minimum scenario: with/without/avoided (avoided uses 5000m attribution)
    axes[0].bar(x_positions - bar_width, rp_df["Min_Damages_Without_Mangroves_USD_mn"], width=bar_width, color="#4C78A8", label="Without mangroves")
    axes[0].bar(x_positions, rp_df["Min_Damages_With_Mangroves_USD_mn"], width=bar_width, color="#F58518", label="With mangroves")
    axes[0].bar(x_positions + bar_width, rp_df["Min_Avoided_Damages_USD_mn"], width=bar_width, color="#1b7837", label=f"Avoided ({mangrove_attribution_buffer_m}m attributed)")
    axes[0].set_title(f"Minimum scenario (RP={int(return_period)})")
    axes[0].set_xticks(x_positions)
    axes[0].set_xticklabels(sectors, rotation=25, ha="right")
    axes[0].set_xlabel("Sector")
    axes[0].set_ylabel("Damages (USD millions)")
    axes[0].yaxis.set_major_formatter(FuncFormatter(money_mn_fmt))
    axes[0].legend(frameon=False, fontsize=8)

    # Maximum scenario: with/without/avoided (avoided uses 5000m attribution)
    axes[1].bar(x_positions - bar_width, rp_df["Max_Damages_Without_Mangroves_USD_mn"], width=bar_width, color="#4C78A8", label="Without mangroves")
    axes[1].bar(x_positions, rp_df["Max_Damages_With_Mangroves_USD_mn"], width=bar_width, color="#F58518", label="With mangroves")
    axes[1].bar(x_positions + bar_width, rp_df["Max_Avoided_Damages_USD_mn"], width=bar_width, color="#1b7837", label=f"Avoided ({mangrove_attribution_buffer_m}m attributed)")
    axes[1].set_title(f"Maximum scenario (RP={int(return_period)})")
    axes[1].set_xticks(x_positions)
    axes[1].set_xticklabels(sectors, rotation=25, ha="right")
    axes[1].set_xlabel("Sector")
    axes[1].set_ylabel("Damages (USD millions)")
    axes[1].yaxis.set_major_formatter(FuncFormatter(money_mn_fmt))
    axes[1].legend(frameon=False, fontsize=8)

    # Percent avoided by sector (5000m-attributed only): minimum vs maximum
    percent_bar_width = 0.38
    axes[2].bar(
        x_positions - percent_bar_width / 2,
        rp_df["Min_Percent_Avoided_vs_NoMangroves"],
        width=percent_bar_width,
        color="#74C476",
        label="Minimum scenario",
    )
    axes[2].bar(
        x_positions + percent_bar_width / 2,
        rp_df["Max_Percent_Avoided_vs_NoMangroves"],
        width=percent_bar_width,
        color="#238B45",
        label="Maximum scenario",
    )
    axes[2].set_title(f"Percent avoided (RP={int(return_period)})")
    axes[2].set_xticks(x_positions)
    axes[2].set_xticklabels(sectors, rotation=25, ha="right")
    axes[2].set_xlabel("Sector")
    axes[2].set_ylabel("Percent avoided (%)")
    y_max = np.nanmax([
        rp_df["Min_Percent_Avoided_vs_NoMangroves"].max(),
        rp_df["Max_Percent_Avoided_vs_NoMangroves"].max(),
    ])
    axes[2].set_ylim(0, max(1, y_max * 1.25))
    axes[2].legend(frameon=False, fontsize=8)

    fig.suptitle(
        f"Sector comparison at return period {int(return_period)} years ({mangrove_attribution_buffer_m}m-attributed avoided)",
        fontsize=14,
    )
    out_png = plot_output_dir / f"return_period_{int(return_period)}_sector_min_max_comparison.png"
    fig.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", out_png)

print("All return-period sector comparison charts saved in:", plot_output_dir)



## Stacked sector shares of avoided damages by return period

Two stacked bars per return period: one for minimum and one for maximum scenario, showing each sector's contribution to total avoided damages.

In [ ]:
return_periods = sorted(rp_sector_compare["ReturnPeriod"].dropna().unique())

for return_period in return_periods:
    rp_df = (
        rp_sector_compare[rp_sector_compare["ReturnPeriod"] == return_period]
        .sort_values("Sector")
        .reset_index(drop=True)
    )

    x_positions = np.arange(2)
    scenario_labels = ["Minimum", "Maximum"]
    bar_width = 0.58

    fig, ax = plt.subplots(figsize=(11.2, 6.4))
    fig.patch.set_facecolor("white")
    ax.set_facecolor("white")

    # Non-blue/green/teal/orange palette.
    sector_colors = {
        "buildings": "#7A1E6C",  # plum
        "energy": "#B22222",     # brick red
        "transport": "#5A3E36",  # cocoa
        "water": "#8A7F80",      # warm gray
    }

    cumulative_bottom = np.zeros(2)
    for _, row in rp_df.iterrows():
        sector_name = str(row["Sector"])
        segment_values = np.array([
            float(row["Min_Avoided_Damages_USD_mn"]),
            float(row["Max_Avoided_Damages_USD_mn"]),
        ])

        ax.bar(
            x_positions,
            segment_values,
            width=bar_width,
            bottom=cumulative_bottom,
            color=sector_colors.get(sector_name, "#6F5F5E"),
            label=sector_name.capitalize(),
            edgecolor="white",
            linewidth=1.0,
            zorder=3,
        )

        for position_index, segment_value in enumerate(segment_values):
            if segment_value <= 0:
                continue
            ax.text(
                x_positions[position_index],
                cumulative_bottom[position_index] + segment_value / 2.0,
                f"{segment_value:,.1f}",
                ha="center",
                va="center",
                fontsize=8 if segment_value < 5 else 9,
                color="white",
                fontweight="bold",
                zorder=5,
            )

        cumulative_bottom += segment_values

    total_values = cumulative_bottom
    total_offset = max(0.6, float(np.nanmax(total_values)) * 0.02)
    for position_index, total_value in enumerate(total_values):
        ax.text(
            x_positions[position_index],
            total_value + total_offset,
            f"Total: {total_value:,.1f}",
            ha="center",
            va="bottom",
            fontsize=10,
            color="#3A3A3A",
            fontweight="bold",
            zorder=6,
        )

    y_top = max(1.0, float(np.nanmax(total_values)) * 1.14)

    ax.set_title(
        f"Total avoided damages with sector composition (RP={int(return_period)})\n{mangrove_attribution_buffer_m}m-attributed only",
        fontsize=13,
        pad=14,
    )
    ax.set_xticks(x_positions)
    ax.set_xticklabels(scenario_labels, fontsize=11)
    ax.set_ylabel("Avoided damages (USD millions)", fontsize=11)
    ax.set_ylim(0, y_top)
    ax.yaxis.set_major_formatter(FuncFormatter(money_mn_fmt))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=8))
    ax.grid(axis="y", linestyle="--", linewidth=0.8, alpha=0.25, zorder=0)
    ax.set_axisbelow(True)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.legend(
        title="Sector",
        frameon=False,
        ncol=4,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.20),
        fontsize=9,
        title_fontsize=9,
        columnspacing=1.4,
        handlelength=1.6,
    )

    fig.subplots_adjust(top=0.78, bottom=0.12)

    out_png = plot_output_dir / f"return_period_{int(return_period)}_sector_avoided_total_stacked_min_max.png"
    fig.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", out_png)

print("All stacked return-period avoided-total charts saved in:", plot_output_dir)

